# 05 - SHAP Analizi

Bu not defteri, modelin kararlarını daha anlaşılır hale getirmek için SHAP yaklaşımını kullanır.

İki düzeyde yorum üretiyoruz:
- Global açıklama: Model genel olarak hangi değişkenlere daha çok bakıyor?
- Lokal açıklama: Tek bir örnek için riski artıran/azaltan etkenler neler?


In [ ]:
# Ortamı hazırlıyor ve proje kökünü Python yoluna ekliyoruz.
from pathlib import Path
import sys

proje_kok = Path.cwd().resolve()
while proje_kok.name != 'diyabet_risk_tahmini' and proje_kok.parent != proje_kok:
    proje_kok = proje_kok.parent

if proje_kok.name != 'diyabet_risk_tahmini':
    raise RuntimeError('Notebook proje kökünden veya alt klasörlerinden çalıştırılmalıdır.')

if str(proje_kok) not in sys.path:
    sys.path.insert(0, str(proje_kok))

print(f'Proje kökü: {proje_kok}')


In [ ]:
# SHAP analizi için gerekli modülleri ve proje fonksiyonlarını içe aktarıyoruz.
import pandas as pd
import matplotlib.pyplot as plt

from makine_ogrenmesi.kaynak.artifact_kaydet import artifactleri_yukle
from makine_ogrenmesi.kaynak.aciklanabilirlik import (
    global_shap_ozeti_hesapla,
    lokal_shap_yorumlari_hesapla,
    global_shap_gorseli_kaydet,
)
from makine_ogrenmesi.kaynak.ozellik_yapilandirmasi import OZELLIK_KOLONLARI
from makine_ogrenmesi.kaynak.veri_yukleyici import veri_setini_yukle


In [ ]:
# Kayıtlı artifact dosyalarını yükleyip açıklama üretiminde kullanılacak modeli alıyoruz.
artifact_klasoru = proje_kok / 'makine_ogrenmesi' / 'artifactler'
artifactler = artifactleri_yukle(artifact_klasoru)

model = artifactler['kalibrator']
metadata = artifactler['model_metadata']

print('Model:', metadata.get('model_adi'))
print('Kalibrasyon yöntemi:', metadata.get('kalibrasyon_yontemi'))
print('Eşik yöntemi:', metadata.get('ikili_siniflama_yontemi'))


In [ ]:
# SHAP hesaplamasında kullanılacak arka plan ve değerlendirme örneklemlerini hazırlıyoruz.
veri = veri_setini_yukle(proje_kok / 'makine_ogrenmesi' / 'veri' / 'ham' / 'diabetes.csv')
X = veri[OZELLIK_KOLONLARI].copy()

x_arka_plan = X.sample(n=min(120, len(X)), random_state=42).reset_index(drop=True)
x_degerlendirme = X.sample(n=min(40, len(X)), random_state=7).reset_index(drop=True)

print('Arka plan boyutu:', x_arka_plan.shape)
print('Değerlendirme boyutu:', x_degerlendirme.shape)


In [ ]:
# Global SHAP etkilerini hesaplayıp önem sıralamasını tablo halinde çıkarıyoruz.
global_ozet = global_shap_ozeti_hesapla(
    model=model,
    x_veri=x_degerlendirme,
    max_arka_plan=40,
    max_degerlendirme=40,
    random_state=42,
)

global_df = pd.DataFrame(global_ozet['global_siralama'])
global_df.head(10)


In [ ]:
# Global SHAP sonuçlarını görsel bir çubuk grafik olarak sunuyoruz.
top10 = global_df.head(10).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10['ozellik'], top10['ortalama_mutlak_shap'], color='#4C78A8')
ax.set_title('Global SHAP - İlk 10 Özellik')
ax.set_xlabel('Ortalama Mutlak SHAP')
ax.grid(axis='x', alpha=0.3)
plt.show()


In [ ]:
# İlk üç örnek için lokal SHAP yorumları üreterek bireysel karar gerekçelerini alıyoruz.
lokal_yorumlar = lokal_shap_yorumlari_hesapla(
    model=model,
    x_arka_plan=x_arka_plan,
    x_ornekler=x_degerlendirme.head(3),
    top_n=3,
    max_arka_plan=40,
    random_state=42,
)

lokal_yorumlar


In [ ]:
# Lokal açıklamaları satır bazında düzenleyip okunabilir bir özet tabloya çeviriyoruz.
lokal_satirlar = []
for yorum in lokal_yorumlar:
    for faktor in yorum['top_faktorler']:
        lokal_satirlar.append({
            'ornek_indeksi': yorum['ornek_indeksi'],
            'tahmin_olasiligi': yorum['tahmin_olasiligi'],
            'ozellik': faktor['ozellik'],
            'ozellik_degeri': faktor['ozellik_degeri'],
            'shap_katkisi': faktor['shap_katkisi'],
            'yon': faktor['yon'],
        })

pd.DataFrame(lokal_satirlar)


In [ ]:
# Sunumda kullanmak üzere global SHAP görselini dosyaya kaydediyoruz.
gorsel_yolu = proje_kok / 'makine_ogrenmesi' / 'raporlar' / 'gorseller' / 'global_shap_ozet.png'
kaydedilen = global_shap_gorseli_kaydet(
    model=model,
    x_veri=x_degerlendirme,
    cikti_yolu=gorsel_yolu,
    max_arka_plan=40,
    max_degerlendirme=40,
    random_state=42,
)
print('Global SHAP görseli kaydedildi:', kaydedilen)


## Kısa değerlendirme

- Global açıklama modelin genel davranışını, lokal açıklama ise tekil karar nedenini gösterir.
- Sunum ve dış paydaş iletişiminde bu iki katmanı birlikte kullanmak güven yaratır.
- Böylece model sadece bir sayı üretmez; kararın arkasındaki hikâyeyi de anlatır.
